In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, mean_squared_error
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv('../data/processed/dataset_80_20_normalized_tcr_with_cohort.csv')

In [25]:
#Variable independiente es immunogenicity y me quedo solo con las variables numericas
X = df.select_dtypes(include=['float64', 'int64']).drop(columns=['immunogenicity'])
y = df['immunogenicity']

In [26]:
#Divido en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [27]:

def optimizar_lr_con_guardado(X, y, n_trials=50):
    def objective(trial):
        params = {
            'class_weight': trial.suggest_categorical('class_weight', 
                                                     ['balanced', {0: 1, 1: 4}, {0: 1, 1: 6}]),
            'penalty': trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet']),
            'C': trial.suggest_float('C', 0.001, 10.0, log=True),
            'max_iter': trial.suggest_int('max_iter', 500, 2000),
            'random_state': 42
        }
        
        # Configurar solver y l1_ratio según el penalty
        if params['penalty'] == 'elasticnet':
            params['solver'] = 'saga'
            params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.1, 0.9)
        elif params['penalty'] == 'l1':
            params['solver'] = 'liblinear'
        else:  # l2
            params['solver'] = trial.suggest_categorical('solver_l2', ['liblinear', 'lbfgs'])
        
        # Crear modelo - pasar TODOS los parámetros incluyendo l1_ratio si existe
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(**params))
        ])
        
        score = cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    # Guardar resultado
    guardar_resultado_optuna_lr(
        'LogisticRegression', 
        study.best_value, 
        study.best_params
    )
    
    return study.best_params, study.best_value

def guardar_resultado_optuna_lr(model_name, result, best_params, filename='../results/reports/Optuna_Results/optuna_results_general.csv'):
    nuevo_resultado = {
        'model': model_name,
        'result': result,
        'best_params': str(best_params)
    }
    
    try:
        df = pd.read_csv(filename)
        df = pd.concat([df, pd.DataFrame([nuevo_resultado])], ignore_index=True)
    except FileNotFoundError:
        df = pd.DataFrame([nuevo_resultado])
    
    df.to_csv(filename, index=False)
    print(f"✅ Resultado guardado: {model_name} - AUC: {result:.4f}")
    
# Función para cargar y usar el mejor modelo (igual que tu sistema)
def cargar_y_entrenar_mejor_lr(X_train, y_train, filename='../results/reports/Optuna_Results/optuna_results_general.csv'):
    df = pd.read_csv(filename)
    
    lr_results = df[df['model'] == 'LogisticRegression'].copy()
    lr_results = lr_results.sort_values('result', ascending=False)
    best_lr_result = lr_results.iloc[0]
    
    print(f"Mejor resultado de LogisticRegression: {best_lr_result['result']}")
    print(f"Parámetros: {best_lr_result['best_params']}")
    
    # Convertir parámetros a diccionario
    best_params = ast.literal_eval(best_lr_result['best_params'])
    
    # Adaptar parámetros - Asegurar solver correcto según penalty
    adapted_params = best_params.copy()
    
    # Configurar solver automáticamente según penalty
    if adapted_params['penalty'] == 'elasticnet':
        adapted_params['solver'] = 'saga'
    elif adapted_params['penalty'] == 'l1':
        adapted_params['solver'] = 'liblinear'
    else:  # l2
        # Usar el solver que estaba en los parámetros o default
        if 'solver_l2' in adapted_params:
            adapted_params['solver'] = adapted_params['solver_l2']
            del adapted_params['solver_l2']
        else:
            adapted_params['solver'] = 'lbfgs'
    
    # Limpiar parámetros que no necesita LogisticRegression
    params_to_remove = ['solver_l2']  # Remover si existe
    for param in params_to_remove:
        adapted_params.pop(param, None)
    
    print(f"Parámetros adaptados: {adapted_params}")
    
    # Crear pipeline final
    model_lr = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(**adapted_params, random_state=42))
    ])
    
    print("Entrenando LogisticRegression con los mejores parámetros...")
    model_lr.fit(X_train, y_train)
    print("¡Modelo entrenado exitosamente!")
    
    return model_lr


In [28]:
#Optimizo y guardo resultados
optimizar_lr_con_guardado(X, y, n_trials=50)

#Cargo y entreno el mejor modelo
import ast
model_lr = cargar_y_entrenar_mejor_lr(X_train, y_train)

[I 2025-11-04 12:51:05,814] A new study created in memory with name: no-name-e1fc0bd9-879b-46f2-bc4f-04625d8cc889
/home/cocdrilo/miniconda3/envs/imblearn/lib/python3.10/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains {0: 1, 1: 4} which is of type dict.
  warnings.warn(message)
/home/cocdrilo/miniconda3/envs/imblearn/lib/python3.10/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains {0: 1, 1: 6} which is of type dict.
  warnings.warn(message)
[I 2025-11-04 12:51:11,815] Trial 0 finished with value: 0.6049051405338148 and parameters: {'class_weight': 'balanced', 'penalty': 'elasticnet', 'C': 0.11112422265171787, 'max_iter': 1377, 'l1_ratio': 0.21270655150875717}. Best is trial 0 with value: 0.6049051405338148.
/home/cocdrilo/m

✅ Resultado guardado: LogisticRegression - AUC: 0.6112
Mejor resultado de LogisticRegression: 0.6112320227743051
Parámetros: {'class_weight': 'balanced', 'penalty': 'elasticnet', 'C': 0.005439396084141765, 'max_iter': 1984, 'l1_ratio': 0.8803466866722778}
Parámetros adaptados: {'class_weight': 'balanced', 'penalty': 'elasticnet', 'C': 0.005439396084141765, 'max_iter': 1984, 'l1_ratio': 0.8803466866722778, 'solver': 'saga'}
Entrenando LogisticRegression con los mejores parámetros...
¡Modelo entrenado exitosamente!


In [29]:
#Hago predicciones
y_pred_lr = model_lr.predict(X_test)
y_proba_lr = model_lr.predict_proba(X_test)[:, 1]

In [30]:
#Evaluo el modelo
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
print(f"ROC AUC Score: {roc_auc_lr:.4f}")
mse_lr = mean_squared_error(y_test, y_proba_lr)
print(f"Mean Squared Error: {mse_lr:.4f}")

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.39      0.53      1637
           1       0.24      0.76      0.37       423

    accuracy                           0.46      2060
   macro avg       0.55      0.57      0.45      2060
weighted avg       0.73      0.46      0.50      2060

Confusion Matrix:
[[ 632 1005]
 [ 102  321]]
ROC AUC Score: 0.6189
Mean Squared Error: 0.2424
